In [1]:
import copy
import os
import torch
from utils.utils_poses.align_traj import align_ate_c2b_use_a2b
import scipy
from utils.utils_poses.comp_ate import compute_rpe, compute_ATE
import numpy as np
from utils.vis_utils import plot_pose
from colmap_parsing_utils import read_images_binary
from pathlib import Path
from nerfstudio.utils.io import load_from_json

In [13]:
def align_pose(pose1, pose2):
    mtx1 = np.array(pose1, dtype=np.double, copy=True)
    mtx2 = np.array(pose2, dtype=np.double, copy=True)

    if mtx1.ndim != 2 or mtx2.ndim != 2:
        raise ValueError("Input matrices must be two-dimensional")
    if mtx1.shape != mtx2.shape:
        raise ValueError("Input matrices must be of same shape")
    if mtx1.size == 0:
        raise ValueError("Input matrices must be >0 rows and >0 cols")

    # translate all the data to the origin
    mtx1 -= np.mean(mtx1, 0)
    mtx2 -= np.mean(mtx2, 0)

    norm1 = np.linalg.norm(mtx1)
    norm2 = np.linalg.norm(mtx2)

    if norm1 == 0 or norm2 == 0:
        raise ValueError("Input matrices must contain >1 unique points")

    # change scaling of data (in rows) such that trace(mtx*mtx') = 1
    mtx1 /= norm1
    mtx2 /= norm2

    # transform mtx2 to minimize disparity
    R, s = scipy.linalg.orthogonal_procrustes(mtx1, mtx2)
    mtx2 = mtx2 * s

    return mtx1, mtx2, R

def eval_pose(poses_gt_c2w,poses_pred_c2w,output_path):
    poses_gt = poses_gt_c2w[:len(poses_pred_c2w)].clone()
    # align scale first (we do this because scale differennt a lot)
    trans_gt_align, trans_est_align, _ = align_pose(poses_gt[:, :3, -1].numpy(),
                                                            poses_pred_c2w[:, :3, -1].numpy())
    poses_gt[:, :3, -1] = torch.from_numpy(trans_gt_align)
    poses_pred_c2w[:, :3, -1] = torch.from_numpy(trans_est_align)

    c2ws_est_aligned = align_ate_c2b_use_a2b(poses_pred_c2w, poses_gt)
    ate = compute_ATE(poses_gt.cpu().numpy(),
                        c2ws_est_aligned.cpu().numpy())
    rpe_trans, rpe_rot = compute_rpe(
        poses_gt.cpu().numpy(), c2ws_est_aligned.cpu().numpy())
    print("RPE_t : {0:.3f}".format(rpe_trans*100),
            '&' "RPE_r : {0:.3f}".format(rpe_rot * 180 / np.pi),
            '&', "ATE : {0:.3f}".format(ate))
    plot_pose(poses_gt.cpu().numpy(), c2ws_est_aligned.cpu().numpy(), output_path)
    # # pdb.set_trace()
    # with open(f"{output_path}/pose_eval.txt", 'w') as f:
    #     f.write("RPE_trans: {:.03f}, RPE_rot: {:.03f}, ATE: {:.03f}".format(
    #         rpe_trans*100,
    #         rpe_rot * 180 / np.pi,
    #         ate))
    #     f.close()



In [14]:
colmap_images_txt="dataset/Tanks/Horse/sparse/0/images.bin"
# camera gt 
images_info=read_images_binary(colmap_images_txt) 


In [15]:
# load estimate camera from ns-export
data_path=Path("outputs/Church/key_gaussian/Key5")
split="train"
meta = load_from_json(data_path / f"transforms_{split}.json")
image_filenames = []
poses = []
for frame in meta:
    fname = data_path / Path(frame["file_path"].replace("./", ""))
    image_filenames.append(fname)
    poses.append(np.array(frame["transform"]))

poses = np.array(poses).astype(np.float32)
pred_pose_c2w = torch.from_numpy(poses[:, :3])  # camera to world transform
from utils.utils_poses.lie_group_helper import convert3x4_4x4
pred_pose_c2w=convert3x4_4x4(pred_pose_c2w)
# pred_pose_c2w=pred_pose_c2w.unsqueeze(0)

In [16]:

eval_pose(pred_pose_c2w,pred_pose_c2w,data_path)

RPE_t : 0.000 &RPE_r : 0.000 & ATE : 0.000
70 poses, 0.465m path length
